# Fase 3 · M08: Auditoría y Cierre

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 3 — Feature Engineering |
| **Módulo** | M08 — Auditoría |

---

## 🎯 Qué hace

Auditoría final de features: detecta columnas constantes, verifica ausencia de leakage, genera el dataset D_strict definitivo limpiado de 23 a 20 columnas.

## 📋 Requisitos

- `data/automl/df_exp_automl_D.parquet`
- `data/automl/df_exp_automl_D_strict.parquet`
- `data/03_features/df_eda_con_target.parquet`

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `data/automl/dataset_final_tfm.parquet` | D_strict final (20 features + target) |
| `data/03_features/validacion/auditoria_features_D_strict.xlsx` | Reporte de auditoría |

## 🔄 Flujo

```
df_exp_automl_D.parquet + df_exp_automl_D_strict.parquet
    ↓ Detección columnas constantes
    ↓ Verificación ausencia leakage
    ↓ Limpieza 23→20 columnas
    → dataset_final_tfm.parquet (D_strict definitivo)
```

## ➡️ Siguiente

Fin de Fase 3 — continuar con `fautoml_m00_indice.ipynb`


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================

import sys, shutil, json as json_mod
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
import time
warnings.filterwarnings('ignore')

ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent


sys.path.insert(0, str(ROOT))

from src.config import RUTA_AUTOML, RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.html import generar_kpis_html, generar_seccion_html, generar_html_navegacion_completa, guardar_html
from src.html.render import render_base_html

RUTA_OLD = RUTA_AUTOML / 'old'
RUTA_FASE3_HTML = RUTA_HTML / 'fase3'
crear_directorios([RUTA_AUTOML, RUTA_FASE3_HTML, RUTA_OLD])
fmt = formato_numero_es
TARGET = 'abandono'

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print('✓ Configuración completada')
info_entorno()

✓ Directorios creados: 1


✓ Configuración completada
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓ =

In [2]:
# ============================================================================
# CELDA 2: DIAGNÓSTICO — Estado actual de D y D_strict
# ============================================================================

print('='*70)
print('DIAGNÓSTICO: ESTADO ACTUAL DE LOS DATASETS')
print('='*70)

# Cargar datasets
df_d = pd.read_parquet(RUTA_AUTOML / 'df_exp_automl_D.parquet')
df_ds = pd.read_parquet(RUTA_AUTOML / 'df_exp_automl_D_strict.parquet')
print(f'Caso D:        {df_d.shape[0]:,} × {df_d.shape[1]} cols ({df_d.shape[1]-1} features)')
print(f'Caso D_strict: {df_ds.shape[0]:,} × {df_ds.shape[1]} cols ({df_ds.shape[1]-1} features)')

# Diagnóstico columna por columna
print(f'\n--- Análisis de variabilidad (D_strict) ---')
print(f'{"#":>3s} {"Columna":30s} {"Únicos":>7s} {"Top%":>6s} {"Nulos%":>7s} {"Diagnóstico"}')
print('-'*85)

# Columnas a eliminar (decisión profesional documentada)
COLS_ELIMINAR = [
    'pago_fraccionado',       # Constante total: 1 único valor, 100%
    'indicador_edad_inusual', # Constante práctica: 100% mismo valor
    'tiene_gaps',             # Redundante con anios_gap (jerárquica)
]

for i, col in enumerate(sorted(df_ds.columns)):
    s = df_ds[col]
    n_u = s.nunique()
    pct_top = (s.value_counts().iloc[0] / len(df_ds)) * 100 if len(s.value_counts()) > 0 else 0
    pct_nulos = s.isna().mean() * 100
    
    if col in COLS_ELIMINAR:
        diag = '🔴 ELIMINAR'
    elif col == TARGET:
        diag = '🎯 TARGET'
    elif pct_top >= 97:
        diag = '🟡 Raro pero predictivo → MANTENER'
    else:
        diag = '✅ OK'
    print(f'{i+1:3d}. {col:30s} {n_u:7d} {pct_top:5.1f}% {pct_nulos:6.1f}%  {diag}')

print(f'\n🗑️ Columnas a eliminar: {COLS_ELIMINAR}')
print(f'   Resultado: {df_ds.shape[1]} - {len(COLS_ELIMINAR)} - 1(target) = {df_ds.shape[1]-len(COLS_ELIMINAR)-1} features')

DIAGNÓSTICO: ESTADO ACTUAL DE LOS DATASETS
Caso D:        33,621 × 25 cols (24 features)
Caso D_strict: 33,621 × 23 cols (22 features)

--- Análisis de variabilidad (D_strict) ---
  # Columna                         Únicos   Top%  Nulos% Diagnóstico
-------------------------------------------------------------------------------------
  1. abandono                             2  70.8%    0.0%  🎯 TARGET
  2. anios_gap                            9  97.0%    0.0%  ✅ OK
  3. anios_sin_beca                      12  36.9%    0.0%  ✅ OK
  4. cred_superados_anio_1er            316  30.1%    0.0%  ✅ OK
  5. edad_entrada                        51  46.1%    0.0%  ✅ OK
  6. indicador_edad_inusual               2 100.0%    0.0%  🔴 ELIMINAR
  7. indicador_interrupcion               2  97.0%    0.0%  ✅ OK
  8. max_pagos                            7  44.6%    0.0%  ✅ OK
  9. n_anios_beca                        10  27.8%    0.0%  ✅ OK
 10. nota_1er_anio                      473   1.0%    7.0%  ✅ OK
 11.

In [3]:
# ============================================================================
# CELDA 3: LIMPIEZA — Eliminar columnas y regenerar datasets
# ============================================================================

print('='*70)
print('LIMPIEZA: ELIMINAR COLUMNAS Y REGENERAR')
print('='*70)

for caso_nombre, df_caso in [('D', df_d), ('D_strict', df_ds)]:
    cols_en_caso = [c for c in COLS_ELIMINAR if c in df_caso.columns]
    if not cols_en_caso:
        print(f'\n  Caso {caso_nombre}: Sin columnas a eliminar ✅')
        continue

    df_limpio = df_caso.drop(columns=cols_en_caso)
    print(f'\n  Caso {caso_nombre}: {df_caso.shape[1]} → {df_limpio.shape[1]} cols')
    print(f'    Eliminadas: {cols_en_caso}')

    # Guardar en 4 formatos
    for ext in ['.parquet', '.csv', '.xlsx', '.json']:
        ruta = RUTA_AUTOML / f'df_exp_automl_{caso_nombre}{ext}'
        if ext == '.parquet': df_limpio.to_parquet(ruta)
        elif ext == '.csv': df_limpio.to_csv(ruta, sep=';', index=False)
        elif ext == '.xlsx': df_limpio.to_excel(ruta, index=False)
        elif ext == '.json': df_limpio.to_json(ruta, orient='records', force_ascii=False)

    if caso_nombre == 'D': df_d = df_limpio
    else: df_ds = df_limpio

print(f'\n📊 Resultado:')
print(f'  Caso D:        {df_d.shape[1]} cols ({df_d.shape[1]-1} features + abandono)')
print(f'  Caso D_strict: {df_ds.shape[1]} cols ({df_ds.shape[1]-1} features + abandono)')
print(f'\n  Columnas D_strict: {sorted([c for c in df_ds.columns if c != TARGET])}')

LIMPIEZA: ELIMINAR COLUMNAS Y REGENERAR

  Caso D: 25 → 22 cols
    Eliminadas: ['pago_fraccionado', 'indicador_edad_inusual', 'tiene_gaps']



  Caso D_strict: 23 → 20 cols
    Eliminadas: ['pago_fraccionado', 'indicador_edad_inusual', 'tiene_gaps']



📊 Resultado:
  Caso D:        22 cols (21 features + abandono)
  Caso D_strict: 20 cols (19 features + abandono)

  Columnas D_strict: ['anios_gap', 'anios_sin_beca', 'cred_superados_anio_1er', 'edad_entrada', 'indicador_interrupcion', 'max_pagos', 'n_anios_beca', 'nota_1er_anio', 'nota_acceso', 'nota_selectividad', 'orden_preferencia', 'pais_nombre', 'provincia', 'rama', 'sexo', 'situacion_laboral', 'tuvo_beca', 'universidad_origen', 'via_acceso']


In [4]:
# ============================================================================
# CELDA 4: MOVER ARCHIVOS CONTAMINADOS A OLD/
# ============================================================================

print('='*70)
print('LIMPIEZA: ARCHIVOS CONTAMINADOS → OLD/')
print('='*70)

patrones_mover = [
    'df_exp_automl_target_final',  # Archivo trampa (Caso A contaminado)
    'df_exp_automl_target.',       # Versión sin 'final'
    'df_exp_automl_A',             # Caso A (leakage)
    'df_exp_automl_B',             # Caso B (leakage)
    'df_exp_eda_A',                # EDA A
    'df_exp_eda_B',                # EDA B
]

movidos = []
for carpeta in [RUTA_AUTOML]:
    for archivo in carpeta.iterdir():
        if archivo.is_file() and archivo.name != '.gitkeep' and archivo.parent.name != 'old':
            for patron in patrones_mover:
                if archivo.name.startswith(patron):
                    destino_dir = carpeta / 'old'
                    destino_dir.mkdir(exist_ok=True)
                    shutil.move(str(archivo), str(destino_dir / archivo.name))
                    movidos.append(archivo.name)
                    print(f'  📦 {archivo.name} → old/')
                    break

if movidos:
    print(f'\n✅ {len(movidos)} archivos movidos a old/')
else:
    print('\n✅ No hay archivos contaminados (ya movidos anteriormente)')

# Crear dataset_final_tfm.parquet = D_strict limpio
ruta_final = RUTA_AUTOML / 'dataset_final_tfm.parquet'
df_ds.to_parquet(ruta_final)
print(f'\n🎯 dataset_final_tfm.parquet creado ({df_ds.shape[1]} cols = D_strict limpio)')

# Listar archivos activos
print(f'\n📁 Archivos activos en data/automl/:')
for f in sorted(RUTA_AUTOML.iterdir()):
    if f.is_file() and f.name != '.gitkeep':
        size_mb = f.stat().st_size / (1024*1024)
        print(f'   {f.name:50s} {size_mb:.1f} MB')

LIMPIEZA: ARCHIVOS CONTAMINADOS → OLD/
  📦 df_exp_automl_A.csv → old/
  📦 df_exp_automl_A.json → old/
  📦 df_exp_automl_A.parquet → old/
  📦 df_exp_automl_A.xlsx → old/
  📦 df_exp_automl_B.csv → old/
  📦 df_exp_automl_B.json → old/
  📦 df_exp_automl_B.parquet → old/
  📦 df_exp_automl_B.xlsx → old/
  📦 df_exp_automl_target_final.parquet → old/

✅ 9 archivos movidos a old/

🎯 dataset_final_tfm.parquet creado (20 cols = D_strict limpio)

📁 Archivos activos en data/automl/:
   auditoria_features_D_strict.xlsx                   0.0 MB
   automl_comparativa_final.json                      0.0 MB
   automl_comparativa_final.xlsx                      0.0 MB
   automl_top_modelos.parquet                         0.0 MB
   comparativa_baselines_D.png                        0.1 MB
   dataset_final_tfm.parquet                          0.3 MB
   df_exp_automl_C.csv                                4.0 MB
   df_exp_automl_C.json                               24.3 MB
   df_exp_automl_C.parquet          

In [5]:
# ============================================================================
# CELDA 5: AUDITORÍA COMPLETA — 19 FEATURES DE D_STRICT
# ============================================================================
# Tres preguntas por variable:
# 1. Consistencia temporal: ¿Se conoce ANTES del abandono?
# 2. Poder predictivo: ¿Explica el porqué (causa) o el qué (consecuencia)?
# 3. Variabilidad: ¿Aporta info nueva o es redundante?

print('='*70)
print('AUDITORÍA COMPLETA — CASO D_STRICT (19 features)')
print('='*70)

# Clasificación experta de cada variable
FICHA_FEATURES = {
    # --- ACADÉMICAS (1er AÑO) ---
    'nota_1er_anio': {
        'categoria': 'Académica (1er año)',
        'disponibilidad': 'Primer año',
        'temporal_ok': True,
        'causal': True,
        'descripcion': 'Nota media del primer año académico',
        'justificacion': 'Predictor #1. Refleja el choque inicial con la carrera.',
    },
    'cred_superados_anio_1er': {
        'categoria': 'Académica (1er año)',
        'disponibilidad': 'Primer año',
        'temporal_ok': True,
        'causal': True,
        'descripcion': 'Créditos superados en el primer año',
        'justificacion': 'Mal inicio académico → mayor riesgo abandono.',
    },
    # --- SOCIOECONÓMICAS ---
    'tuvo_beca': {
        'categoria': 'Socioeconómica',
        'disponibilidad': 'Primer año',
        'temporal_ok': True,
        'causal': True,
        'descripcion': 'Indicador de si tuvo beca alguna vez (0/1)',
        'justificacion': 'Factor crítico: sin beca → más presión económica.',
    },
    'n_anios_beca': {
        'categoria': 'Socioeconómica',
        'disponibilidad': 'Primer año',
        'temporal_ok': True,
        'causal': True,
        'descripcion': 'Número de años con beca',
        'justificacion': 'Apoyo económico sostenido reduce abandono.',
    },
    'anios_sin_beca': {
        'categoria': 'Socioeconómica',
        'disponibilidad': 'Primer año',
        'temporal_ok': True,
        'causal': True,
        'descripcion': 'Número de años sin beca',
        'justificacion': 'Años sin apoyo económico → acumulación de riesgo.',
    },
    'situacion_laboral': {
        'categoria': 'Socioeconómica',
        'disponibilidad': 'Matrícula',
        'temporal_ok': True,
        'causal': True,
        'descripcion': 'Situación laboral al matricularse',
        'justificacion': 'Alumnos que trabajan → mayor conflicto horario y abandono.',
    },
    'max_pagos': {
        'categoria': 'Socioeconómica',
        'disponibilidad': 'Observable',
        'temporal_ok': True,
        'causal': True,
        'descripcion': 'Número máximo de pagos en un curso',
        'justificacion': 'Refleja capacidad financiera y carga económica.',
    },
    # --- PERFIL DE ENTRADA ---
    'nota_acceso': {
        'categoria': 'Perfil entrada',
        'disponibilidad': 'Matrícula',
        'temporal_ok': True,
        'causal': True,
        'descripcion': 'Nota de acceso a la universidad',
        'justificacion': '¿Tenía nivel académico suficiente?',
    },
    'nota_selectividad': {
        'categoria': 'Perfil entrada',
        'disponibilidad': 'Matrícula',
        'temporal_ok': True,
        'causal': True,
        'descripcion': 'Nota de selectividad/PAAU',
        'justificacion': 'Preparación académica previa.',
    },
    'via_acceso': {
        'categoria': 'Perfil entrada',
        'disponibilidad': 'Matrícula',
        'temporal_ok': True,
        'causal': True,
        'descripcion': 'Vía de acceso (PAAU, FP, >25...)',
        'justificacion': '¿Entró por la vía convencional o alternativa?',
    },
    'orden_preferencia': {
        'categoria': 'Perfil entrada',
        'disponibilidad': 'Matrícula',
        'temporal_ok': True,
        'causal': True,
        'descripcion': 'Orden de preferencia en preinscripción',
        'justificacion': '¿Entró en lo que quería o donde pudo?',
    },
    'universidad_origen': {
        'categoria': 'Perfil entrada',
        'disponibilidad': 'Matrícula',
        'temporal_ok': True,
        'causal': False,
        'descripcion': 'Universidad de procedencia',
        'justificacion': 'Contexto de origen.',
    },
    'rama': {
        'categoria': 'Perfil entrada',
        'disponibilidad': 'Matrícula',
        'temporal_ok': True,
        'causal': True,
        'descripcion': 'Rama de conocimiento de la titulación',
        'justificacion': 'Algunas ramas tienen más abandono estructural.',
    },
    # --- DEMOGRÁFICAS ---
    'sexo': {
        'categoria': 'Demográfica',
        'disponibilidad': 'Matrícula',
        'temporal_ok': True,
        'causal': False,
        'descripcion': 'Sexo del alumno (0=H, 1=M)',
        'justificacion': 'Factor demográfico, no causal directo.',
    },
    'edad_entrada': {
        'categoria': 'Demográfica',
        'disponibilidad': 'Matrícula',
        'temporal_ok': True,
        'causal': True,
        'descripcion': 'Edad al matricularse por primera vez',
        'justificacion': 'Edad inusual puede indicar mayor responsabilidad o dificultad.',
    },
    'pais_nombre': {
        'categoria': 'Demográfica',
        'disponibilidad': 'Matrícula',
        'temporal_ok': True,
        'causal': False,
        'descripcion': 'País de origen del alumno',
        'justificacion': 'Contexto cultural y de movilidad.',
    },
    'provincia': {
        'categoria': 'Demográfica',
        'disponibilidad': 'Matrícula',
        'temporal_ok': True,
        'causal': False,
        'descripcion': 'Provincia de domicilio',
        'justificacion': 'Distancia al campus como factor logístico.',
    },
    # --- TRAYECTORIA ---
    'anios_gap': {
        'categoria': 'Trayectoria',
        'disponibilidad': 'Observable',
        'temporal_ok': True,
        'causal': True,
        'descripcion': 'Años de gap (sin matrícula)',
        'justificacion': 'Evento raro (3%) pero muy predictivo de abandono.',
    },
    'indicador_interrupcion': {
        'categoria': 'Trayectoria',
        'disponibilidad': 'Observable',
        'temporal_ok': True,
        'causal': True,
        'descripcion': 'Si hubo interrupción formal en la matrícula (0/1)',
        'justificacion': 'Interrupción formal ≠ gap. Anticipa abandono.',
    },
}

# Generar tabla de auditoría
auditoria = []
for col in sorted(df_ds.columns):
    s = df_ds[col]
    ficha = FICHA_FEATURES.get(col, {})
    fila = {
        'columna': col,
        'tipo': str(s.dtype),
        'n_unicos': s.nunique(),
        'pct_nulos': round(s.isna().mean()*100, 2),
        'min': round(float(s.min()), 2) if s.dtype in ['int64','float64'] else '-',
        'max': round(float(s.max()), 2) if s.dtype in ['int64','float64'] else '-',
        'media': round(float(s.mean()), 4) if s.dtype in ['int64','float64'] else '-',
        'std': round(float(s.std()), 4) if s.dtype in ['int64','float64'] else '-',
        'categoria': ficha.get('categoria', 'Target' if col == TARGET else '?'),
        'disponibilidad': ficha.get('disponibilidad', 'Target' if col == TARGET else '?'),
        'temporal_ok': ficha.get('temporal_ok', True),
        'causal': ficha.get('causal', False),
        'descripcion': ficha.get('descripcion', 'Target binario' if col == TARGET else ''),
        'justificacion': ficha.get('justificacion', ''),
    }
    auditoria.append(fila)

df_audit = pd.DataFrame(auditoria)

# Mostrar tabla
print(f'\n📊 TABLA DE AUDITORÍA ({len(df_audit)} columnas):\n')
print(f'{"Columna":30s} {"Categoría":20s} {"Cuándo":12s} {"Temporal":>8s} {"Causal":>7s} {"Únicos":>7s} {"Nulos%":>7s}')
print('-'*100)
for _, row in df_audit.iterrows():
    temp = '✅' if row['temporal_ok'] else '❌'
    caus = '✅' if row['causal'] else '—'
    print(f'{row["columna"]:30s} {row["categoria"]:20s} {row["disponibilidad"]:12s} {temp:>8s} {caus:>7s} {row["n_unicos"]:7d} {row["pct_nulos"]:6.1f}%')

# Verificaciones de seguridad
print(f'\n\n🔍 VERIFICACIONES DE SEGURIDAD:')
print('-'*50)
checks = {
    'Columnas constantes': [c for c in df_ds.columns if c!=TARGET and df_ds[c].nunique()<=1],
    'Leakage directo': [c for c in ['egresado','egresado_de_hecho','curso_ultimo','anos_inactivo','pct_titulacion','cred_faltantes','per_id_ficticio','exp_tit_id'] if c in df_ds.columns],
    'Temporales': [c for c in ['curso_inicio','duracion_real','n_cursos'] if c in df_ds.columns],
    'Traidoras': [c for c in ['indicador_sin_notas','nota_ultimo_anio','cred_superados_total','cred_matriculados_total','cred_superados_anio_medio','tasa_rendimiento','ratio_avance','velocidad_creditos','progreso_relativo','media_global','cred_titulacion','estabilidad_academica','mejora_notas'] if c in df_ds.columns],
    'Redundantes': [c for c in ['pago_fraccionado','indicador_edad_inusual','tiene_gaps'] if c in df_ds.columns],
    '>95% nulos': [c for c in df_ds.columns if df_ds[c].isna().mean()>0.95],
    'Temporal_ok=False': [r['columna'] for _,r in df_audit.iterrows() if not r['temporal_ok'] and r['columna']!=TARGET],
}
all_ok = True
for check, resultado in checks.items():
    ok = len(resultado) == 0
    print(f'  {"✅" if ok else "❌"} {check}: {resultado if resultado else "Ninguna"}')
    if not ok: all_ok = False

n_aband = (df_ds[TARGET]==1).sum()
print(f'  ✅ Target: abandono=1 → {n_aband:,} ({n_aband/len(df_ds)*100:.1f}%)')
print(f'\n{"✅ AUDITORÍA SUPERADA" if all_ok else "⚠️ HAY PROBLEMAS"}')

AUDITORÍA COMPLETA — CASO D_STRICT (19 features)

📊 TABLA DE AUDITORÍA (20 columnas):

Columna                        Categoría            Cuándo       Temporal  Causal  Únicos  Nulos%
----------------------------------------------------------------------------------------------------
abandono                       Target               Target              ✅       —       2    0.0%
anios_gap                      Trayectoria          Observable          ✅       ✅       9    0.0%
anios_sin_beca                 Socioeconómica       Primer año          ✅       ✅      12    0.0%
cred_superados_anio_1er        Académica (1er año)  Primer año          ✅       ✅     316    0.0%
edad_entrada                   Demográfica          Matrícula           ✅       ✅      51    0.0%
indicador_interrupcion         Trayectoria          Observable          ✅       ✅       2    0.0%
max_pagos                      Socioeconómica       Observable          ✅       ✅       7    0.0%
n_anios_beca                

In [6]:
# ============================================================================
# CELDA 6: DICCIONARIO DE COLUMNAS (JSON + Excel)
# ============================================================================

print('='*70)
print('DICCIONARIO MAESTRO DE COLUMNAS')
print('='*70)

# JSON
diccionario = {}
for _, row in df_audit.iterrows():
    diccionario[row['columna']] = {
        'tipo': row['tipo'], 'n_unicos': int(row['n_unicos']),
        'pct_nulos': float(row['pct_nulos']),
        'categoria': row['categoria'],
        'disponibilidad': row['disponibilidad'],
        'temporal_ok': bool(row['temporal_ok']),
        'causal': bool(row['causal']),
        'descripcion': row['descripcion'],
        'justificacion': row['justificacion'],
        'leakage': 'NO'
    }

ruta_json = RUTA_AUTOML / 'diccionario_columnas.json'
with open(ruta_json, 'w', encoding='utf-8') as f:
    json_mod.dump(diccionario, f, ensure_ascii=False, indent=2, default=str)
print(f'💾 {ruta_json.name}')

# Excel
ruta_xlsx = RUTA_AUTOML / 'auditoria_features_D_strict.xlsx'
df_audit.to_excel(ruta_xlsx, index=False)
print(f'💾 {ruta_xlsx.name}')
print(f'\n📊 {len(diccionario)} columnas documentadas')

DICCIONARIO MAESTRO DE COLUMNAS
💾 diccionario_columnas.json
💾 auditoria_features_D_strict.xlsx

📊 20 columnas documentadas


In [7]:
# ============================================================================
# CELDA 7: EL GRÁFICO DE LA VERDAD
# ============================================================================

print('='*70)
print('EL GRÁFICO DE LA VERDAD')
print('='*70)

data_verdad = {
    'Caso': ['Caso A\n(Leakage)', 'Caso D\n(Alerta Temprana)', 'Caso D_strict\n(Producción)'],
    'AUC': [1.00, 0.9433, 0.9288],
    'F1': [1.00, 0.8002, 0.7821]
}
df_verdad = pd.DataFrame(data_verdad)

fig, ax = plt.subplots(figsize=(12, 7))
x = np.arange(3)
width = 0.35

colores_auc = ['#e53e3e', '#3182ce', '#38a169']
colores_f1 = ['#fc8181', '#90cdf4', '#9ae6b4']

bars1 = ax.bar(x - width/2, df_verdad['AUC'], width, label='AUC-ROC',
    color=colores_auc, edgecolor='white', linewidth=2)
bars2 = ax.bar(x + width/2, df_verdad['F1'], width, label='F1-Score',
    color=colores_f1, edgecolor='white', linewidth=2)

for bar in bars1:
    h = bar.get_height()
    ax.text(bar.get_x()+bar.get_width()/2, h+0.01, f'{h:.2f}', ha='center', fontweight='bold', fontsize=12)
for bar in bars2:
    h = bar.get_height()
    ax.text(bar.get_x()+bar.get_width()/2, h+0.01, f'{h:.2f}', ha='center', fontweight='bold', fontsize=11)

ax.annotate('MODELO INÚTIL\n(Sobreajuste por Fuga)', xy=(0, 1.02),
    fontsize=11, color='#e53e3e', ha='center', fontweight='bold')
ax.annotate('MODELO DE PRODUCCIÓN\n(Alerta Temprana Real)', xy=(2, 0.95),
    fontsize=11, color='#38a169', ha='center', fontweight='bold')

ax.axhline(y=0.95, color='#a0aec0', linestyle='--', alpha=0.5)
ax.set_ylabel('Puntuación', fontsize=13, fontweight='bold')
ax.set_title('El Gráfico de la Verdad:\nImpacto de la Auditoría de Datos en el Rendimiento',
    fontsize=15, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(df_verdad['Caso'], fontweight='bold', fontsize=11)
ax.set_ylim(0, 1.15)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.2)

fig.text(0.5, 0.01,
    'Figura: Comparativa de Rendimiento — Detección de Fuga de Información (Leakage) vs. Modelo de Producción Realista',
    ha='center', fontsize=10, style='italic', color='#4a5568')

plt.tight_layout(rect=[0, 0.04, 1, 1])
ruta_grafico = RUTA_AUTOML / 'grafico_de_la_verdad.png'
fig.savefig(str(ruta_grafico), dpi=200, bbox_inches='tight')
plt.show()
plt.close()
print(f'💾 {ruta_grafico.name}')

EL GRÁFICO DE LA VERDAD


💾 grafico_de_la_verdad.png


In [8]:
# ============================================================================
# CELDA 8: HTML
# ============================================================================

import base64

nav_fases, nav_modulos = generar_html_navegacion_completa(fase_activa='fase3', modulo_activo='m08')

n_feat = df_ds.shape[1] - 1
KPIS = [
    {'valor': str(n_feat), 'titulo': 'Features'},
    {'valor': '0', 'titulo': 'Leakage'},
    {'valor': '0', 'titulo': 'Constantes'},
    {'valor': '✅', 'titulo': 'Auditoría'},
]

contenido = ''

# Tabla de auditoría
tabla = '<table style="width:100%;border-collapse:collapse;font-size:12px;">\n'
tabla += '<tr style="background:#2d3748;color:white;">'
for h in ['Columna','Categoría','Cuándo','Temporal','Causal','Únicos','Nulos%','Descripción']:
    tabla += f'<th style="padding:6px;">{h}</th>'
tabla += '</tr>\n'
for i, (_, row) in enumerate(df_audit.iterrows()):
    bg = '#f7fafc' if i%2 else 'white'
    temp = '✅' if row['temporal_ok'] else '❌'
    caus = '✅' if row['causal'] else '—'
    tabla += f'<tr style="background:{bg};">'
    tabla += f'<td style="padding:4px;font-family:monospace;font-weight:bold;">{row["columna"]}</td>'
    tabla += f'<td>{row["categoria"]}</td>'
    tabla += f'<td>{row["disponibilidad"]}</td>'
    tabla += f'<td style="text-align:center;">{temp}</td>'
    tabla += f'<td style="text-align:center;">{caus}</td>'
    tabla += f'<td style="text-align:center;">{row["n_unicos"]}</td>'
    tabla += f'<td style="text-align:center;">{row["pct_nulos"]}%</td>'
    tabla += f'<td style="font-size:11px;">{row["descripcion"]}</td>'
    tabla += '</tr>\n'
tabla += '</table>'
contenido += generar_seccion_html('📊 Auditoría de Features — Las 19 Variables', tabla)

# Gráfico
if ruta_grafico.exists():
    with open(ruta_grafico, 'rb') as f: b64 = base64.b64encode(f.read()).decode()
    img = f'<img src="data:image/png;base64,{b64}" style="max-width:100%;border-radius:8px;">'
    img += '<p style="font-style:italic;color:#4a5568;margin-top:10px;">'
    img += 'Figura: Comparativa de Rendimiento — Detección de Fuga de Información vs. Modelo de Producción Realista</p>'
    contenido += generar_seccion_html('🎯 El Gráfico de la Verdad', img)

# Checklist
checklist = '''<div style="background:#f0fff4;padding:20px;border-radius:10px;">\n'''
checklist += '<h3>Checklist Auditoría Features</h3>\n'
checklist += '<p>✅ Columnas constantes eliminadas: pago_fraccionado, indicador_edad_inusual</p>\n'
checklist += '<p>✅ Columna redundante eliminada: tiene_gaps (redundante con anios_gap)</p>\n'
checklist += '<p>✅ Archivos contaminados (A, B) movidos a old/</p>\n'
checklist += '<p>✅ dataset_final_tfm.parquet creado = D_strict limpio</p>\n'
checklist += '<p>✅ Leakage directo: Ninguno</p>\n'
checklist += '<p>✅ Variables temporales: Ninguna</p>\n'
checklist += '<p>✅ Variables traidoras: Ninguna</p>\n'
checklist += '<p>✅ 19 features: todas disponibles antes del abandono</p>\n'
checklist += '<p>✅ Diccionario de columnas generado (JSON + Excel)</p>\n'
checklist += '</div>'
contenido += generar_seccion_html('✅ Checklist Final', checklist)

# Columnas eliminadas en todo el proceso
eliminadas = '''<table style="width:100%;border-collapse:collapse;">\n'''
eliminadas += '<tr style="background:#e53e3e;color:white;"><th style="padding:8px;">Columna</th><th>Motivo</th><th>Eliminada en</th></tr>\n'
elim_lista = [
    ('egresado', 'Leakage directo (es el target)', 'M05'),
    ('egresado_de_hecho', 'Leakage directo', 'M05'),
    ('curso_ultimo', 'Leakage directo (fórmula target)', 'M05'),
    ('anos_inactivo', 'Leakage directo', 'M05'),
    ('pct_titulacion', 'Leakage directo', 'M05'),
    ('cred_faltantes', 'Leakage directo', 'M05'),
    ('per_id_ficticio', 'ID', 'M05'),
    ('exp_tit_id', 'ID', 'M05'),
    ('indicador_casi_termino', 'Leakage indirecto', 'M05 (B)'),
    ('curso_inicio', 'Temporal (reconstruye target)', 'M05 (C)'),
    ('duracion_real', 'Temporal (reconstruye target)', 'M05 (C)'),
    ('n_cursos', 'Temporal (reconstruye target)', 'M05 (C)'),
    ('indicador_sin_notas', 'Traidora: sin notas = ya abandonó', 'M06 (D)'),
    ('nota_ultimo_anio', 'Traidora: nota 0 = abandonó', 'M06 (D)'),
    ('cred_superados_total', 'Traidora: acumula toda la historia', 'M06 (D)'),
    ('cred_matriculados_total', 'Traidora: acumula toda la historia', 'M06 (D)'),
    ('cred_superados_anio_medio', 'Traidora: necesita años totales', 'M06 (D)'),
    ('tasa_rendimiento', 'Traidora: acumula todo', 'M06 (D)'),
    ('ratio_avance', 'Traidora: foto del final', 'M06 (D)'),
    ('velocidad_creditos', 'Traidora: temporal', 'M06 (D)'),
    ('progreso_relativo', 'Traidora: foto del final', 'M06 (D)'),
    ('media_global', 'Traidora: usa notas posteriores', 'M06 (D)'),
    ('cred_titulacion', 'Constante, no aporta', 'M06 (D)'),
    ('estabilidad_academica', 'Traidora: toda trayectoria', 'M06 (D_strict)'),
    ('mejora_notas', 'Traidora: notas posteriores', 'M06 (D_strict)'),
    ('pago_fraccionado', 'Constante total (1 valor)', 'M08'),
    ('indicador_edad_inusual', 'Constante práctica (100%)', 'M08'),
    ('tiene_gaps', 'Redundante con anios_gap', 'M08'),
]
for col, motivo, donde in elim_lista:
    eliminadas += f'<tr><td style="padding:6px;font-family:monospace;">{col}</td><td>{motivo}</td><td>{donde}</td></tr>\n'
eliminadas += '</table>'
contenido += generar_seccion_html('🗑️ Historial de Columnas Eliminadas (28 en total)', eliminadas)

html = render_base_html(titulo='M08: Auditoría y Cierre Fase 3', icono='🔍',
    subtitulo='Fase 3: Feature Engineering — Cierre Definitivo',
    nav_fases=nav_fases, nav_modulos=nav_modulos,
    contenido=generar_kpis_html(KPIS) + contenido,
    notebook_nombre='f3_m08_auditoria.ipynb', notebook_carpeta='fase3_features')
ruta_html = RUTA_FASE3_HTML / 'm08_auditoria.html'
guardar_html(html, ruta_html)
print(f'✅ HTML: {ruta_html}')

✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase3\m08_auditoria.html
✅ HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase3\m08_auditoria.html


In [9]:
# ============================================================================
# CELDA 9: EXPORTAR df_eda_final A data/04_eda/ (4 formatos)
# ============================================================================
# Fuente: df_eda_con_target.parquet (generado por M05)
#   - Ya tiene target (abandono) calculado
#   - Ya tiene leakage eliminado
#   - Categoricas en TEXTO LEGIBLE
#
# Seleccion: 19 features auditadas + titulacion + target = 21 cols
# TRAZABILIDAD: M04b > M05 (target) > M08 (auditoria) > 04_eda/
# ============================================================================

print('='*70)
print('EXPORTAR DATASET DEFINITIVO PARA FASE 4 (EDA FINAL)')
print('='*70)

RUTA_EDA = ROOT / 'data' / '04_eda'
crear_directorios([RUTA_EDA])

# -----------------------------------------------------------------
# Cargar fuente: EDA con target (generado por M05)
# -----------------------------------------------------------------
RUTA_FEATURES = ROOT / 'data' / '03_features'
df_fuente = pd.read_parquet(RUTA_FEATURES / 'df_eda_con_target.parquet')
print(f'\n  Fuente: df_eda_con_target.parquet')
print(f'  Forma:  {df_fuente.shape[0]:,} x {df_fuente.shape[1]} cols')
print(f'  Target: {(df_fuente[TARGET].mean()*100):.1f}% abandono')

# -----------------------------------------------------------------
# Seleccionar: 19 features auditadas + titulacion + target
# -----------------------------------------------------------------
features_auditadas = sorted([c for c in df_ds.columns if c != TARGET])
COLS_EDA = features_auditadas + ['titulacion', TARGET]

faltantes = [c for c in COLS_EDA if c not in df_fuente.columns]
if faltantes:
    print(f'\n  ERROR: columnas no encontradas: {faltantes}')
    print(f'  Disponibles: {sorted(df_fuente.columns.tolist())}')
    raise ValueError(f'Faltan columnas: {faltantes}')

df_eda_final = df_fuente[COLS_EDA].copy()

# -----------------------------------------------------------------
# Verificacion de integridad contra D_strict
# -----------------------------------------------------------------
assert df_eda_final.shape[0] == df_ds.shape[0], \
    f'Filas: EDA={df_eda_final.shape[0]} vs D_strict={df_ds.shape[0]}'
assert (df_eda_final[TARGET].values == df_ds[TARGET].values).all(), \
    'El target no coincide entre versiones'
print(f'\n  Integridad OK: {df_eda_final.shape[0]:,} filas, target identico')

# -----------------------------------------------------------------
# Exportar en 4 formatos
# -----------------------------------------------------------------
nombre = 'df_eda_final'
df_eda_final.to_parquet(RUTA_EDA / f'{nombre}.parquet', index=False)
df_eda_final.to_csv(RUTA_EDA / f'{nombre}.csv', sep=';', index=False)
df_eda_final.to_excel(RUTA_EDA / f'{nombre}.xlsx', index=False)
df_eda_final.to_json(RUTA_EDA / f'{nombre}.json', orient='records', force_ascii=False, indent=2)

cats_texto = df_eda_final.select_dtypes(include='object').columns.tolist()
print(f'\n  df_eda_final:')
print(f'    {df_eda_final.shape[0]:,} filas x {df_eda_final.shape[1]} columnas')
print(f'    19 features + titulacion + abandono')
print(f'    Variables texto: {cats_texto}')
print(f'    Titulaciones: {df_eda_final["titulacion"].nunique()}')
print(f'\n  Destino: data/04_eda/ (4 formatos)')
print(f'  Generado por: f3_m08_auditoria.ipynb (Celda 9)')



EXPORTAR DATASET DEFINITIVO PARA FASE 4 (EDA FINAL)
✓ Directorios verificados: 1

  Fuente: df_eda_con_target.parquet
  Forma:  33,621 x 41 cols
  Target: 29.2% abandono

  Integridad OK: 33,621 filas, target identico



  df_eda_final:
    33,621 filas x 21 columnas
    19 features + titulacion + abandono
    Variables texto: ['pais_nombre', 'provincia', 'rama', 'sexo', 'situacion_laboral', 'universidad_origen', 'via_acceso', 'titulacion']
    Titulaciones: 40

  Destino: data/04_eda/ (4 formatos)
  Generado por: f3_m08_auditoria.ipynb (Celda 9)


In [10]:
# ============================================================================
# CELDA 10: RESUMEN FINAL — FASE 3 CERRADA
# ============================================================================

print('\n' + '='*70)
print('✅ F3-M08 AUDITORÍA COMPLETADA — FASE 3 CERRADA DEFINITIVAMENTE')
print('='*70)

print(f'\n🗑️ COLUMNAS ELIMINADAS EN M08:')
print(f'   - pago_fraccionado (constante total)')
print(f'   - indicador_edad_inusual (constante práctica)')
print(f'   - tiene_gaps (redundante con anios_gap)')

print(f'\n📊 DATASET DEFINITIVO PARA FASE 4 (EDA FINAL):')
print(f'   ┌──────────────────────────────────────────────────────────────┐')
print(f'   │ data/04_eda/df_eda_final.parquet (.csv .xlsx .json)         │')
print(f'   │   → 21 cols (19 features + titulacion + abandono)           │')
print(f'   │   → Categóricas con texto legible                           │')
print(f'   │   → Uso: Fase 4 (EDA Final) + Tableau + Power BI           │')
print(f'   └──────────────────────────────────────────────────────────────┘')

print(f'\n📋 LAS 19 FEATURES AUDITADAS:')
cats = {}
for _, row in df_audit.iterrows():
    if row['columna'] == TARGET: continue
    cat = row['categoria']
    if cat not in cats: cats[cat] = []
    cats[cat].append(row['columna'])
for cat, cols in sorted(cats.items()):
    print(f'   {cat}: {", ".join(cols)}')

print(f'\n   + titulacion (40 grados, para EDA y Streamlit)')

print(f'\n✅ VERIFICACIONES:')
print(f'   Leakage: NO | Temporales: NO | Traidoras: NO')
print(f'   Constantes: NO | Redundantes: NO | >95%% nulos: NO')
print(f'   Todas las features disponibles ANTES del abandono: SÍ')

print(f'\n📁 ARCHIVOS GENERADOS POR ESTE NOTEBOOK:')
for f in ['diccionario_columnas.json', 'auditoria_features_D_strict.xlsx',
          'grafico_de_la_verdad.png', 'dataset_final_tfm.parquet',
          'm08_auditoria.html',
          'df_eda_final.parquet', 'df_eda_final.csv',
          'df_eda_final.xlsx', 'df_eda_final.json']:
    print(f'   {f}')


print(f'\n🎯 FASE 3 CERRADA.')
print(f'   Siguiente: Fase 4 (EDA Final) con data/04_eda/df_eda_final.parquet')
print(f'   El dataset para Fase 5 (Modelado) se generará en f4_m08_conclusiones.ipynb')
print('='*70)



✅ F3-M08 AUDITORÍA COMPLETADA — FASE 3 CERRADA DEFINITIVAMENTE

🗑️ COLUMNAS ELIMINADAS EN M08:
   - pago_fraccionado (constante total)
   - indicador_edad_inusual (constante práctica)
   - tiene_gaps (redundante con anios_gap)

📊 DATASET DEFINITIVO PARA FASE 4 (EDA FINAL):
   ┌──────────────────────────────────────────────────────────────┐
   │ data/04_eda/df_eda_final.parquet (.csv .xlsx .json)         │
   │   → 21 cols (19 features + titulacion + abandono)           │
   │   → Categóricas con texto legible                           │
   │   → Uso: Fase 4 (EDA Final) + Tableau + Power BI           │
   └──────────────────────────────────────────────────────────────┘

📋 LAS 19 FEATURES AUDITADAS:
   Académica (1er año): cred_superados_anio_1er, nota_1er_anio
   Demográfica: edad_entrada, pais_nombre, provincia, sexo
   Perfil entrada: nota_acceso, nota_selectividad, orden_preferencia, rama, universidad_origen, via_acceso
   Socioeconómica: anios_sin_beca, max_pagos, n_anios_beca, sit